# NB1 Data Prep - the Gentrification's real-estate dimension 

This notebook builds the real-estate dimension of gentrification. Data is loaded, transformed into usable csv and re-coded on the correct level (PLR, "Planungsraum" - planning area). 

In addition, the features are cleaned for the purpose of the next NB2 data prep - the EDA of the real-estate dimension of gentrification. 

In [1]:
#import libraries

import pandas as pd
import pdfplumber
import geopandas as gpd
import os
import numpy as np
import openpyxl as op
from pathlib import Path 


## Variable 1 and 2: rent prices over time and as state in 2025

This cell opens the PDF version of the IBB asking-rents and inspects the first page. It then defines regex patterns for the year headers and the data rows, verifies them on test cases, and loops over all pages from 2021 onward to collect PLR ID, name, year, median, case count, and mean into a list. 

The data can be found here: https://www.ibb.de/media/dokumente/publikationen/berliner-wohnungsmarkt/wohnungsmarktbericht/2025/ibb-wohnungsmarktbericht-angebotsmieten_2012-2025.pdf



In [2]:
#open PDF and test first page 
PDF_PFAD = "../../data/real_estate/raw/IBB_Angebotsmieten_2012-2025.pdf"   

with pdfplumber.open(PDF_PFAD) as pdf:
    print("Anzahl Seiten:", len(pdf.pages))
    seite = pdf.pages[2]              
    text = seite.extract_text()
    print(text[:1000])

#yearly code for finding the header 
import re
jahr_muster = re.compile(r"Median (20\d\d)")

#Muster erstellen damit Zeilen extrahiert werden können 
zeile_muster = re.compile(
    r"^(\d{8})\s+(.+?)\s+"
    r"(keine Daten|\d{1,2},\d{2})\s+"            # median
    r"(keine Daten|\d+)\s+"                       # fallzahl
    r"(keine Daten|#NV|\d{1,2},\d{2})\s*$"        # mittel: also allow #NV
)

print(zeile_muster)

#testen ob Muster passt 
test = "02500726 Hausburgviertel 18,70 67 18,11"
treffer = zeile_muster.match(test)
print(treffer.groups() if treffer else "kein Treffer")

test2 = "03200204 Blankenfelde keine Daten keine Daten keine Daten"
treffer2 = zeile_muster.match(test2)
print(treffer2.groups() if treffer2 else "kein Treffer")

#Sammelschleife für Variablen basteln 
zeilen = []
nicht_erkannt = []
seiten_verarbeitet = 0
zeilen_geprueft = 0

with pdfplumber.open(PDF_PFAD) as pdf:
    aktuelles_jahr = None
    for i, seite in enumerate(pdf.pages):
        text = seite.extract_text() or ""
        treffer_jahr = jahr_muster.search(text)
        if treffer_jahr:
            aktuelles_jahr = int(treffer_jahr.group(1))
        if aktuelles_jahr is None or aktuelles_jahr < 2021:
            continue
        seiten_verarbeitet += 1
        for zeile in text.split("\n"):
            zeile = zeile.strip()
            if re.match(r"^\d{8}\s", zeile):       
                zeilen_geprueft += 1
                treffer = zeile_muster.match(zeile)
                if treffer:
                    g = treffer.groups()
                    zeilen.append({"plr_id": g[0], "plr_name": g[1].strip(),
                                   "jahr": aktuelles_jahr, "median": g[2],
                                   "fallzahl": g[3], "mittel": g[4]})
                else:
                    nicht_erkannt.append((aktuelles_jahr, zeile))

print("Seiten verarbeitet:", seiten_verarbeitet)
print("Zeilen geprüft:", zeilen_geprueft)
print("Treffer gesammelt:", len(zeilen))
print("Datenzeilen, die NICHT gematcht wurden:", len(nicht_erkannt))
for j, z in nicht_erkannt[:20]:
    print(j, repr(z))



Anzahl Seiten: 153
02500726 Hausburgviertel 18,70 67 18,11
02500727 Forckenbeckplatz 18,00 28 18,92
02500728 Samariterviertel 17,96 76 19,92
02500729 Pettenkofer Straße 21,73 48 21,06
02500830 Niederbarnimstraße 18,50 26 18,62
02500831 Wismarplatz 15,98 40 16,16
02500832 Traveplatz 21,06 95 20,99
02500833 Boxhagener Platz 20,15 106 20,38
02500834 Revaler Straße 20,97 73 21,88
02500835 Stralauer Kiez 21,83 92 20,92
02500836 Stralauer Halbinsel 19,32 39 19,45
03100101 Bucher Forst 15,50 44 14,86
03100102 Klinikum Buch 9,78 43 11,43
03100103 Karower Chaussee 7,77 78 8,00
03200204 Blankenfelde keine Daten keine Daten keine Daten
03200205 Uhlandstraße 13,70 54 14,73
03200206 Pastor-Niemöller-Platz 14,30 36 15,38
03200207 Herthaplatz 15,45 76 15,75
03200308 Nisbléstraße 13,89 42 14,36
03200309 Eddastraße 14,30 28 15,36
03200310 Krugpfuhl 14,79 67 15,02
03300411 Karow Nord 10,98 127 11,99
03300412 Karow Bahnhof 13,00 12 15,36
03300413 Karow Ost keine Daten keine Daten keine Daten
03300514 Bla

In [3]:
#build a dataFrame and control for 542 observations 

df_miete = pd.DataFrame(zeilen)
print(df_miete.shape)
print(df_miete.groupby("jahr")["plr_id"].nunique())

#what happens in Blankenfelde?
ids_2021 = set(df_miete[df_miete["jahr"]==2021]["plr_id"])
ids_2022 = set(df_miete[df_miete["jahr"]==2022]["plr_id"])
print("In 2021 aber nicht 2022:", ids_2021 - ids_2022)

#Blankenfelde is a very low densed PLR
print(df_miete[df_miete["plr_id"] == "03200204"])

#Cleaning: German strings -> numeric, "keine Daten" -> NaN
#rent: comma -> point, then to numeric (no Data will automatically be NaN)
df_miete["median"] = pd.to_numeric(df_miete["median"].astype(str).str.replace(",", ".", regex=False), errors="coerce")

#number of cases: whole figures (keine Daten wird NaN)
df_miete["fallzahl"] = pd.to_numeric(df_miete["fallzahl"], errors="coerce").astype("Int64")

# drop mittel — using median as the rent measure as it is more accurate for rent data 
df_miete = df_miete.drop(columns="mittel")

#test
print(df_miete.dtypes)
print(df_miete["median"].describe())
print("NaN in median:", df_miete["median"].isna().sum())

#save raw form, one line per PLR-yes 
df_miete.to_csv("../../data/real_estate/raw/var1_angebotsmieten_2021_2025_lang.csv", index=False)

#new form with plr_id as observation 
wide = df_miete.pivot_table(index="plr_id", columns="jahr",
                      values=["median", "fallzahl"], aggfunc="first")
wide.columns = [f"{wert}_{jahr}" for wert, jahr in wide.columns]
wide = wide.reset_index()

#full PLR-list as a base
alle_plr = df_miete[["plr_id", "plr_name"]].drop_duplicates("plr_id")

#attach plr_name (pivot dropped it; merge does NOT change the row count, stays 542)
wide = alle_plr.merge(wide, on="plr_id", how="left")

print(wide.shape)   
print("PLR ohne Mietdaten 2025:", wide["median_2025"].isna().sum())

for j in [2021, 2022, 2023, 2024, 2025]:
    fehlt = wide[f"median_{j}"].isna().sum()
    print(f"{j}: {fehlt} PLR ohne Wert")


(2710, 6)
jahr
2021    542
2022    542
2023    542
2024    542
2025    542
Name: plr_id, dtype: int64
In 2021 aber nicht 2022: set()
        plr_id      plr_name  jahr       median     fallzahl       mittel
88    03200204  Blankenfelde  2025  keine Daten  keine Daten  keine Daten
630   03200204  Blankenfelde  2024        14,14            8        16,03
1172  03200204  Blankenfelde  2023  keine Daten  keine Daten  keine Daten
1714  03200204  Blankenfelde  2022  keine Daten  keine Daten          #NV
2256  03200204  Blankenfelde  2021        12,18            9        11,77
plr_id          str
plr_name        str
jahr          int64
median      float64
fallzahl      Int64
dtype: object
count    2596.000000
mean       13.101059
std         4.450620
min         5.370000
25%         9.630000
50%        12.590000
75%        16.135000
max        29.430000
Name: median, dtype: float64
NaN in median: 114
(542, 12)
PLR ohne Mietdaten 2025: 20
2021: 25 PLR ohne Wert
2022: 24 PLR ohne Wert
2023: 27 

In [4]:
#Werte-Plausibilität pro Jahr — fängt Spaltenverschiebungen
print(df_miete.groupby("jahr")["median"].describe()[["min", "50%", "max"]])
#konkret: alles außerhalb realistischer Berliner Angebotsmieten
ausreisser = df_miete[(df_miete["median"] < 5) | (df_miete["median"] > 40)]
print(f"\n{len(ausreisser)} verdächtige Median-Werte:")
print(ausreisser[["plr_id", "plr_name", "jahr", "median", "fallzahl"]])

       min    50%    max
jahr                    
2021  5.76  10.49  22.56
2022  5.37  11.14  27.08
2023  5.88  13.00  26.85
2024  6.29  14.50  29.43
2025  6.58  14.86  27.50

0 verdächtige Median-Werte:
Empty DataFrame
Columns: [plr_id, plr_name, jahr, median, fallzahl]
Index: []


In [5]:
dup = df_miete[df_miete.duplicated(["plr_id", "jahr"], keep=False)]
print(f"Doppelte plr_id/jahr-Kombinationen: {len(dup)}")
print(dup.sort_values(["plr_id", "jahr"]))

Doppelte plr_id/jahr-Kombinationen: 0
Empty DataFrame
Columns: [plr_id, plr_name, jahr, median, fallzahl]
Index: []


In [6]:
#crosscheck the missing PLRs per year 
print("Rows per year:")
print(df_miete.groupby("jahr")["plr_id"].nunique())
print("\nTotal NaN in median:", df_miete["median"].isna().sum())

fehlend = (df_miete[df_miete["median"].isna()]
           .groupby("plr_id").size())
print("PLR ohne Daten in ALLEN 5 Jahren:", (fehlend == 5).sum())


Rows per year:
jahr
2021    542
2022    542
2023    542
2024    542
2025    542
Name: plr_id, dtype: int64

Total NaN in median: 114
PLR ohne Daten in ALLEN 5 Jahren: 11


In [7]:
#How many PLRs have a complete five-year series?
median_cols = [f"median_{j}" for j in range(2021, 2026)]
print("PLR with complete 2021-2025 series:", wide[median_cols].notna().all(axis=1).sum())
print("PLR with at least one gap:", wide[median_cols].isna().any(axis=1).sum())

PLR with complete 2021-2025 series: 501
PLR with at least one gap: 41


In [8]:
#final crosschecks after building the Dataset 0 (the base for the models)

alle_bekannt = set(pd.read_csv("../../data/final_datasets/dataset_0.csv",
                               dtype={"plr_id": str})["plr_id"].str.zfill(8))
pdf_ids = set(wide["plr_id"])
print("Zeilen in wide:", len(wide), "(erwartet: bis zu 542)")
print("IDs im PDF, aber unbekannt:", pdf_ids - alle_bekannt)   
print("Bekannte PLR ohne PDF-Eintrag:", len(alle_bekannt - pdf_ids))

print("Zeilen in dataset_0:", len(alle_bekannt))   
print("PDF-PLR gesamt:", df_miete["plr_id"].nunique())  
print("Differenz (im PDF, nicht in dataset_0):", len(set(df_miete["plr_id"]) - alle_bekannt))  

#only the 15 dropped planning areas were found. Double check with dataset 0 successful. 



Zeilen in wide: 542 (erwartet: bis zu 542)
IDs im PDF, aber unbekannt: {'03300515', '10100205', '03300413', '08200728', '06200418', '12200411', '05200419', '04200207', '03400831', '12200414', '05300838', '08200729', '04400725', '03300517', '09100306'}
Bekannte PLR ohne PDF-Eintrag: 0
Zeilen in dataset_0: 527
PDF-PLR gesamt: 542
Differenz (im PDF, nicht in dataset_0): 15


### Result - Data Load Rent Variables 2021 - 2025

The PDF has 153 pages of which 50 (2021 onward) were included: the regex parses normal, "keine Daten," and #NV rows, and all 2,710 checked data rows matched with zero unrecognized. Beyond match-rate, the extraction was validated on content: no duplicate PLR-year combinations, and per-year median rents fall within a plausible Berlin range (≈5–30 €/m²) with no spurious outliers, ruling out column-shift parse errors. The median rises steadily across years (10.49 → 14.86 €/m² from 2021 to 2025), consistent with the real citywide asking-rent increase. 

All 542 PLR are captured: a cross-check against dataset_0 (527 PLR) leaves a difference of exactly 15, which are the PLR excluded when dataset_0 was built.

Every year holds all 542 PLR as rows, so the extraction is complete: all 2,710 expected PLR-years (542 × 5) are captured, with both source "no value" conventions (keine Daten, #NV) accepted so no valid row is dropped.

Missing values are therefore genuine source gaps stored as NaN, not lost rows. In total 114 PLR-years carry no rent figure (2021: 25, 2022: 24, 2023: 27, 2024: 18, 2025: 20). Eleven PLR have no data in any year — typically very low-density or non-residential areas such as Karow Ost, Blankenburger Süden, and Güterbahnhof Grunewald. These structural gaps cannot be filled and should be flagged or excluded at clustering. The decision how to handle planning areas that only miss individual years and retain partial data, is decided going forward.

In the next cells, the variable 1 and 2 are engineered to measure both current state of rent in Berlin and the process of increasing rents. 

In [9]:
#Engineering var 1 + 2 level of the youngest available rent + process as a trend, relative to Bezirk and to Berlin as a city
#aggregate districts from planning areas  
wide["bez"] = wide["plr_id"].str[:2]

#variable 1 the current level 
median_cols = ["median_2025", "median_2024", "median_2023", "median_2022", "median_2021"]
wide["miete_niveau"] = wide[median_cols].bfill(axis=1).iloc[:, 0]

print("Zustand fehlt bei:", wide["miete_niveau"].isna().sum(), "PLR")
print(wide[["plr_id", "miete_niveau"]].head()) 


#Variable 2 the trend 
jahre = np.array([2021, 2022, 2023, 2024, 2025])
trend_cols = ["median_2021", "median_2022", "median_2023", "median_2024", "median_2025"]

def trend(zeile):
    werte = zeile[trend_cols].values.astype(float)
    maske = ~np.isnan(werte)
    if maske.sum() < 2:
        return np.nan
    return np.polyfit(jahre[maske], werte[maske], 1)[0]

wide["miete_trend"] = wide.apply(trend, axis=1)

print(wide["miete_trend"].describe())

#Make both relative to Bezirk and control for amount of ads 

#first control, z-score only from trustworthy PLRs 
fallzahl_cols = ["fallzahl_2021","fallzahl_2022","fallzahl_2023","fallzahl_2024","fallzahl_2025"]
wide["fallzahl_mittel"] = wide[fallzahl_cols].mean(axis=1)
wide["miete_unsicher"] = (
    (wide["fallzahl_mittel"] < 21) | wide["fallzahl_mittel"].isna()
).astype(int)

#create a verlaesslich variable, to check for amount of adds / PLR, 21 adds / PLR is the measurement from IBB which we take over 
verlaesslich = wide["miete_unsicher"] == 0
for c in ["miete_niveau", "miete_trend"]:
    # vs Bezirk
    grp = wide.loc[verlaesslich].groupby("bez")[c]
    wide[f"{c}_vs_bez"] = (wide[c] - wide["bez"].map(grp.mean())) / wide["bez"].map(grp.std())
    # vs Berlin
    mean_berlin = wide.loc[verlaesslich, c].mean()
    std_berlin  = wide.loc[verlaesslich, c].std()
    wide[f"{c}_vs_berlin"] = (wide[c] - mean_berlin) / std_berlin


print(wide[["miete_niveau_vs_bez", "miete_trend_vs_bez", "fallzahl_mittel", "miete_unsicher"]].describe())
print(wide[["miete_niveau_vs_berlin", "miete_trend_vs_berlin", "fallzahl_mittel", "miete_unsicher"]].describe())
print("\nunsichere PLR (Fallzahl < 21):", wide["miete_unsicher"].sum())

#save the result 
os.makedirs("../../data/real_estate/final_variables", exist_ok=True)

#take wide form with median and amount of ads per year - prep for var 4
wide.to_csv("../../data/real_estate/raw/var1_angebotsmieten_2021_2025.csv", index=False)

#only take relevant columns 
spalten = ["plr_id", "plr_name", "bez",
           "miete_niveau", "miete_niveau_vs_bez", "miete_niveau_vs_berlin",
           "miete_trend", "miete_trend_vs_bez", "miete_trend_vs_berlin",
           "fallzahl_mittel", "miete_unsicher"]

var1 = wide[spalten].copy()
var1.to_csv("../../data/real_estate/final_variables/var1_2_angebotsmieten.csv", index=False)
print("gespeichert:", var1.shape)
var1.head()



Zustand fehlt bei: 11 PLR
     plr_id  miete_niveau
0  01100101         22.70
1  01100102         20.30
2  01100103         23.00
3  01100104         26.21
4  01100205         26.08
count    526.000000
mean       1.094531
std        0.755275
min       -1.510000
25%        0.567000
50%        0.997000
75%        1.519500
max        5.047000
Name: miete_trend, dtype: float64
       miete_niveau_vs_bez  miete_trend_vs_bez  fallzahl_mittel  \
count           531.000000          526.000000            531.0   
mean             -0.045360           -0.034353         42.24253   
std               0.994402            1.041680        30.256978   
min              -2.859758           -4.406496              3.0   
25%              -0.815082           -0.736764             21.0   
50%              -0.009717           -0.164885             35.8   
75%               0.695061            0.486508             54.2   
max               3.233911            5.604011            216.2   

       miete_unsiche

,plr_id,plr_name,bez,miete_niveau,miete_niveau_vs_bez,miete_niveau_vs_berlin,miete_trend,miete_trend_vs_bez,miete_trend_vs_berlin,fallzahl_mittel,miete_unsicher
0,01100101,Stülerstraße,01,22.70,1.020478,1.650489,2.142,1.224713,1.380585,23.8,0
1,01100102,Großer Tiergarten,01,20.30,0.483545,1.123496,1.059,-0.391013,-0.102295,58.4,0
2,01100103,Lützowstraße,01,23.00,1.087595,1.716363,1.356,0.052081,0.304367,58.2,0
3,01100104,Körnerstraße,01,26.21,1.805742,2.421217,2.276,1.424627,1.564063,39.8,0
4,01100205,Wilhelmstraße,01,26.08,1.776659,2.392672,1.583,0.390742,0.615183,64.4,0


In [10]:
#check zscore denominators 
print(wide.loc[verlaesslich].groupby("bez").size().min())

23


The check is sound: every district has at least 24 reliable PLRs, most have more.

In [11]:
#check for usable values in niveau and trend 
usable = wide[trend_cols].notna().sum(axis=1)
print((usable == 0).sum(), (usable == 1).sum())  

11 5


### Result - Building Variable 1 and 2 

The rent level ("miete_niveau") is the most recent available median per PLR, taken via backfill across 2021–2025, and is missing only for the 11 chronically empty PLR. Because it is the newest available figure, it does not always refer to the same year — mostly 2025, but 2024 or earlier where recent years are missing. Given broadly rising rents, levels from earlier years are slightly understated relative to 2025-based ones — a minor inconsistency to note at the clustering stage.

The trend ("miete_trend"), the slope of median rent over time, is available for 526 PLR and mostly positive (mean ≈ 1.09 €/m² per year), reflecting rising asking rents citywide. It requires at least two years to fit, so it is undefined for 16 PLR (the 11 fully-empty plus five with only one usable year).

Both values are standardised twice: relative to their district (vs_bez) and relative to Berlin as a whole (vs_berlin), in each case using means and standard deviations computed only from reliable PLR. The district view captures how a planning area stands against its immediate neighbours; the city view captures how it stands against Berlin overall. Both behave as expected — means near 0 and standard deviations near 1 (Berlin level: −0.08 / 0.97; Berlin trend: −0.05 / 1.03; district level: −0.05 / 0.99; district trend: −0.03 / 1.04), consistent with correctly applied standardisation; small deviations from exactly 0/1 arise because the reference statistics use only reliable PLR while the z-scores are applied to all PLR.

A reliability flag ("miete_unsicher") marks planning areas resting on a thin listing base. The threshold of 21 listings is adopted from the data publisher IBB as the minimum for a robust median; on this basis 131 PLR — almost a quarter of all 542 — are flagged. 

This is the central caveat for the next steps: these values are less stable and should be weighted or flagged rather than treated equally. The final table keeps all 542 PLR, preserving the flagged PLR rather than dropping them.

## Variable 3: Vacancy rate

The next few cells read the census vacancy table from excel, cleans the column names, and filters out header and total rows. The numbers are stripped of German thousands separators and placeholders, from which the vacancy rate (vacant dwellings divided by total stock) is computed and saved as CSV with an eight-digit PLR ID.

The data can be found here: https://www.statistik-berlin-brandenburg.de/zensus22/lokale-daten-berlin

At the very bottom the whole census on Berlins' planning area level can be downloaded. This document is used to build several variables. Where possible the original raw dataset was used. 

In [12]:
# Variable 3 — Leerstandsquote (Zensus 2022 structural vacancy)

PFAD = "../../data/real_estate/raw/Zensus2022vollständigPLR.xlsx"

# Read; header is in row index 1, names contain newlines
roh = pd.read_excel(PFAD, sheet_name="4.9 Wohnungen", header=None, skiprows=9)
df_leer = pd.DataFrame({
    "planungsraum":      roh.iloc[:, 0],
    "wohnungen_gesamt":  roh.iloc[:, 1],
    "leer":              roh.iloc[:, 5],
}).copy()

# Keep only real PLR rows (drop header / Insgesamt-summary rows)
df_leer = df_leer[df_leer["planungsraum"].astype(str).str.match(r"^\d+$")].reset_index(drop=True)

# Clean German-formatted numbers: strip thousands separators, map dashes/slashes to NaN
def zu_zahl(s):
    s = s.astype(str).str.strip().str.replace(".", "", regex=False)
    s = s.replace({"–": np.nan, "-": np.nan, "/": np.nan, "": np.nan})
    return pd.to_numeric(s, errors="coerce")

df_leer["wohnungen_gesamt"] = zu_zahl(df_leer["wohnungen_gesamt"])
df_leer["leer"] = zu_zahl(df_leer["leer"])

# Vacancy rate, stored directly as percentage points (×100 folded in → idempotent)
df_leer["leerstandsquote"] = df_leer["leer"] / df_leer["wohnungen_gesamt"] * 100

# Guards: catch a scaling regression early (Berlin Zensus vacancy ≈ 2%).
# gesamtquote is computed from raw counts, so it stays a FRACTION → band 0.015–0.030.
gesamtquote = df_leer["leer"].sum() / df_leer["wohnungen_gesamt"].sum()
assert 0.015 < gesamtquote < 0.030, f"Gesamtquote {gesamtquote:.4f} außerhalb des Plausibilitätsbereichs"
assert df_leer["wohnungen_gesamt"].iloc[0] == 2065, "Skalierungsfehler: wohnungen_gesamt stimmt nicht"

print("rows:", df_leer.shape[0])
print("Berlin-Gesamtquote:", round(gesamtquote, 4))
print(df_leer["leerstandsquote"].describe())   

# Save
import os
os.makedirs("../../data/real_estate/final_variables", exist_ok=True)
spalten = ["planungsraum", "wohnungen_gesamt", "leer", "leerstandsquote"]
var3 = df_leer[spalten].rename(columns={"planungsraum": "plr_id"})
var3["plr_id"] = var3["plr_id"].astype(str).str.zfill(8)
var3.to_csv("../../data/real_estate/final_variables/var3_leerstandsquote.csv", index=False)
print("gespeichert:", var3.shape)
var3.head()

rows: 541
Berlin-Gesamtquote: 0.0201
count    537.000000
mean       2.093405
std        1.248768
min        0.361197
25%        1.315213
50%        1.778154
75%        2.465294
max        9.523810
Name: leerstandsquote, dtype: float64
gespeichert: (541, 4)


/Users/lauragies/Documents/Job/NF_Bootcamp/capstone-kiezkeeper/.venv/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,plr_id,wohnungen_gesamt,leer,leerstandsquote
0,01100101,2065.0,63.0,3.050847
1,01100102,1163.0,53.0,4.557180
2,01100103,2897.0,54.0,1.863997
3,01100104,2498.0,43.0,1.721377
4,01100205,1362.0,61.0,4.478708


### Result Data Load vacancy rate 2022

The vacancy rate (leerstandsquote) is derived from the Zensus 2022 dwelling-use breakdown as vacant dwellings over total dwellings per PLR, stored in percentage points. It averages about 2.1% (median 1.8%, max 9.5%), which is low but plausible for Berlin's tight housing market. 

These figures reflect the narrow census definition of structural vacancy at a single reference date and should not be compared directly with higher market-vacancy series, which capture frictional and short-term vacancy the Zensus excludes.

Coverage is near-complete: 541 of 542 PLR are present (one PLR is absent from the source), and 537 carry a computed rate, with four PLR lacking a value due to suppressed cells. These gaps are genuine source gaps, carried forward as NaN rather than imputed.

## Variable 4 Rent Gap 2022

These cells load the census existing-tenancy rent, cleans it, and joins it to the 2025 asking rent from Variable 1 via the eight-digit PLR ID (outer join, so no PLR is lost). The rent gap is computed from the difference, checked for extreme and negative values via a plausibility check, and saved as CSV. 

The census data is found here again: https://www.statistik-berlin-brandenburg.de/zensus22/lokale-daten-berlin


In [13]:
#Variable 4 — Mietlücke (rent gap): Angebotsmiete 2025 vs. Zensus Bestandsmiete 2022
#note: existing tenancy rent is only available in census 2022, asking rents however also for 2025
#However, the choice of gap is justifiable - see below in results 

#Header sits on row 5; row 6 is the Berlin "Insgesamt" summary; data from row 7
#Single clean 2-column sheet, so header=5 works 
bestand = pd.read_excel(PFAD, sheet_name="4.16 Wohnungen", header=5)
bestand = bestand.rename(columns={
    "Planungsraum": "plr_id",
    "Mittelwert (WOHN_QM_MIETE)": "bestandsmiete",
})

# Keep only real PLR rows (drops the "Insgesamt" summary + any trailing notes)
bestand = bestand[bestand["plr_id"].astype(str).str.match(r"^\d+$")].reset_index(drop=True)

# German decimal comma -> float; dashes/slashes/blanks -> NaN
def zu_zahl(s):
    s = s.astype(str).str.strip().str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    s = s.replace({"–": np.nan, "-": np.nan, "/": np.nan, "": np.nan, "nan": np.nan})
    return pd.to_numeric(s, errors="coerce")

bestand["bestandsmiete"] = zu_zahl(bestand["bestandsmiete"])
bestand["plr_id"] = bestand["plr_id"].astype(str).str.zfill(8)

# Guards: Berlin Insgesamt was 7,67; PLR-level mean should land near that.
assert 6.0 < bestand["bestandsmiete"].mean() < 9.0, \
    f"Bestandsmiete-Mittel {bestand['bestandsmiete'].mean():.2f} außerhalb des Plausibilitätsbereichs"

print("rows:", bestand.shape[0])
print(bestand["bestandsmiete"].describe())
bestand.head()

rows: 541
count    540.000000
mean       7.785537
std        1.135290
min        5.060000
25%        7.060000
50%        7.720000
75%        8.420000
max       13.580000
Name: bestandsmiete, dtype: float64


/Users/lauragies/Documents/Job/NF_Bootcamp/capstone-kiezkeeper/.venv/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,plr_id,bestandsmiete
0,01100101,10.12
1,01100102,11.87
2,01100103,8.62
3,01100104,8.42
4,01100205,10.05


### Result Variable 4 Rent Gap 

The rent gap is asking rent 2025 minus existing-tenancy rent (Zensus Bestandsmiete 2022), spanning roughly three years. It averages about €7.03/m² (median €6.89), with existing-tenancy rents around €7.79/m² against substantially higher asking rents — confirming that asking rents broadly exceed sitting-tenant rents. The largest gaps (up to €19.21/m²) fall in central, high-demand PLR, while only two PLR show marginally negative gaps (both effectively zero) in cheaper areas. The measure covers 522 PLR, with 20 NaN where either rent figure is missing. 

Although the Bestand side is dated 2022, it matches the September-2025 Mietspiegel 2026 (€7.71/m²) to within €0.05, so contract rents barely moved and the gap reflects mainly genuine asking-vs-contract spread rather than time drift.

## Variable 5 Land Value Housing 2021 - 2025

These cells load the standard land values per year via the WFS service from the Geodata Berlin protal and assigns them to the planning areas through an area-weighted spatial overlay, separated by use group (residential, commercial, mixed/core, other). 

A coverage degree is also computed per PLR, and all years are merged onto the master data and saved as a single CSV. This spatial aggregation is needed because land values come as polygons rather than at PLR level.

The data can be found here: https://daten.berlin.de/datensaetze/bodenrichtwerte-01-01-2025-wfs-7ca7f2c3

In [14]:
#Load data — retains per-land-use coverage per PLR
#WFS gdi.berlin.de currently not available 5th July
#cell works unless the portal is down, however raw data are in the geodata folder 
WORKING_CRS = 25833
JAHRE = [2021, 2022, 2023, 2024, 2025]

def gruppe(n):
    n = str(n)
    if n.startswith("GB"): return "sonstige"   
    if n.startswith("W"):  return "wohnen"
    if n.startswith("G"):  return "gewerbe"
    if n.startswith("M"):  return "misch_kern"
    return "sonstige"

def brw_jahr_zu_plr(jahr, plr, plr_area):
    url = (f"https://gdi.berlin.de/services/wfs/brw{jahr}"
           f"?service=WFS&version=2.0.0&request=GetFeature"
           f"&typeNames=brw{jahr}:brw_{jahr}_vector"
           f"&srsName=EPSG:25833")
    brw = gpd.read_file(url).to_crs(WORKING_CRS)
    brw["brw"] = pd.to_numeric(brw["brw"], errors="coerce")
    brw = brw[brw["brw"].notna()]
    brw["geometry"] = brw.geometry.buffer(0)
    brw["nutz_gruppe"] = brw["nutzung"].map(gruppe)

    inter = gpd.overlay(brw[["brw", "nutz_gruppe", "geometry"]],
                        plr[["plr_id", "geometry"]],
                        how="intersection", keep_geom_type=True)
    inter["area"] = inter.geometry.area
    inter = inter[inter["area"] > 0]
    inter["w"] = inter["brw"] * inter["area"]

    agg = (inter.groupby(["plr_id", "nutz_gruppe"])
           .agg(w_sum=("w", "sum"), area_sum=("area", "sum")))
    agg["brw"] = agg["w_sum"] / agg["area_sum"]

    #area-weighted BRW per group, wide
    wide = agg["brw"].unstack("nutz_gruppe").add_prefix("brw_").reset_index()

    #per-GROUP coverage = group area / PLR area (NEW)
    grp_area = agg["area_sum"].unstack("nutz_gruppe")
    grp_cov = grp_area.div(plr_area, axis=0).clip(upper=1.0)
    grp_cov = grp_cov.add_prefix("cov_").reset_index()
    wide = wide.merge(grp_cov, on="plr_id", how="left")

    #total coverage (as before)
    gedeckt = inter.groupby("plr_id")["area"].sum()
    wide["coverage"] = wide["plr_id"].map((gedeckt / plr_area).clip(upper=1.0))

    #append year to all value columns
    wide = wide.rename(columns={c: f"{c}_{jahr}" for c in wide.columns if c != "plr_id"})
    return wide

#PLR + Stammdaten
plr = gpd.read_file("../../data/real_estate/Geodata/PLR_raw.geojson").to_crs(WORKING_CRS)
plr["geometry"] = plr.geometry.buffer(0)
plr_area = plr.set_index("plr_id").geometry.area   # computed once
stamm = plr[["plr_id", "plr_name", "bzr_name", "bez"]].copy()

gesamt = stamm.copy()
for jahr in JAHRE:
    print(f"Verarbeite {jahr} ...")
    df_jahr = brw_jahr_zu_plr(jahr, plr, plr_area)   # let errors raise, no silent skip
    gesamt = gesamt.merge(df_jahr, on="plr_id", how="left")
    print(f"  {jahr}: ok")

#guard: all five years present (each year adds its block of columns)
assert sum(c.endswith("_2021") for c in gesamt.columns) > 0 and \
       sum(c.endswith("_2025") for c in gesamt.columns) > 0, "Ein Jahr fehlt!"

out = "../../data/real_estate/final_variables/var5_brw_2021_2025_plr.csv"
gesamt.to_csv(out, index=False)
print(f"\nGespeichert: {gesamt.shape} -> {out}")
print(gesamt.columns.tolist())

Verarbeite 2021 ...
  2021: ok
Verarbeite 2022 ...
  2022: ok
Verarbeite 2023 ...
  2023: ok
Verarbeite 2024 ...
  2024: ok
Verarbeite 2025 ...
  2025: ok

Gespeichert: (542, 49) -> ../../data/real_estate/final_variables/var5_brw_2021_2025_plr.csv
['plr_id', 'plr_name', 'bzr_name', 'bez', 'brw_gewerbe_2021', 'brw_misch_kern_2021', 'brw_sonstige_2021', 'brw_wohnen_2021', 'cov_gewerbe_2021', 'cov_misch_kern_2021', 'cov_sonstige_2021', 'cov_wohnen_2021', 'coverage_2021', 'brw_gewerbe_2022', 'brw_misch_kern_2022', 'brw_sonstige_2022', 'brw_wohnen_2022', 'cov_gewerbe_2022', 'cov_misch_kern_2022', 'cov_sonstige_2022', 'cov_wohnen_2022', 'coverage_2022', 'brw_gewerbe_2023', 'brw_misch_kern_2023', 'brw_sonstige_2023', 'brw_wohnen_2023', 'cov_gewerbe_2023', 'cov_misch_kern_2023', 'cov_sonstige_2023', 'cov_wohnen_2023', 'coverage_2023', 'brw_gewerbe_2024', 'brw_misch_kern_2024', 'brw_sonstige_2024', 'brw_wohnen_2024', 'cov_gewerbe_2024', 'cov_misch_kern_2024', 'cov_sonstige_2024', 'cov_wohnen_20

In [15]:
#build the variable housing-brw niveau and trend 

brw = gesamt.copy()
brw["plr_id"] = brw["plr_id"].astype(str).str.zfill(8)

def baue_niveau_trend(df, nutzung, prefix):
    for j in [2021, 2025]:
        df[f"brw_{nutzung}_{j}"] = pd.to_numeric(df[f"brw_{nutzung}_{j}"], errors="coerce")
    df[f"{prefix}_niveau"] = df[f"brw_{nutzung}_2025"]
    a = df[f"brw_{nutzung}_2021"]
    b = df[f"brw_{nutzung}_2025"]
    # log-ratio trend ln(2025/2021), only where both > 0
    df[f"{prefix}_trend"] = np.where((a > 0) & (b > 0), np.log(b / a), np.nan)

def relationen(df, prefix, ref_niveau, ref_trend):
    #niveau: standardise against the full valid pool
    c = f"{prefix}_niveau"
    m, s = df.loc[ref_niveau, c].mean(), df.loc[ref_niveau, c].std()
    df[f"{c}_vs_berlin"] = (df[c] - m) / s
    grp = df.loc[ref_niveau].groupby("bez")[c]
    df[f"{c}_vs_bez"] = (df[c] - df["bez"].map(grp.mean())) / df["bez"].map(grp.std())
    #trend: standardise against reliable, non-outlier PLR
    c = f"{prefix}_trend"
    m, s = df.loc[ref_trend, c].mean(), df.loc[ref_trend, c].std()
    df[f"{c}_vs_berlin"] = (df[c] - m) / s
    grp = df.loc[ref_trend].groupby("bez")[c]
    df[f"{c}_vs_bez"] = (df[c] - df["bez"].map(grp.mean())) / df["bez"].map(grp.std())

#WOHNEN (only land use carried forward, as Gewerbe didnt have enough coverage per PLR)
baue_niveau_trend(brw, "wohnen", "var5_brw_wohnen")

#reliability: residential land covers >= 50% of the PLR (2025)
verl_wohnen = brw["cov_wohnen_2025"].fillna(0) >= 0.5

#outlier flag on the log-trend (|z| > 3 within reliable pool)
t = brw["var5_brw_wohnen_trend"]
tm, ts = t[verl_wohnen].mean(), t[verl_wohnen].std()
brw["var5_brw_wohnen_trend_outlier"] = ((t - tm).abs() > 3 * ts).astype("Int64")

#reference pools: niveau and trend both from the full valid pool
#reliability/outlier status is carried as separate flag columns, not used here
ref_niveau_w = brw["var5_brw_wohnen_niveau"].notna()
ref_trend_w  = brw["var5_brw_wohnen_trend"].notna()
relationen(brw, "var5_brw_wohnen", ref_niveau_w, ref_trend_w)

print("Valid niveau PLR:", int(ref_niveau_w.sum()))
print("Reliable PLR (cov>=0.5):", int(verl_wohnen.sum()))
print("Trend reference PLR (reliable, no outlier):", int(ref_trend_w.sum()))
print("Trend outliers:", int(brw["var5_brw_wohnen_trend_outlier"].sum()))
print("\nStd-Check (sollte ~1.0 sein):")
print(brw[["var5_brw_wohnen_niveau_vs_bez", "var5_brw_wohnen_niveau_vs_berlin",
           "var5_brw_wohnen_trend_vs_bez", "var5_brw_wohnen_trend_vs_berlin"]].std())
print("\nWOHNEN:")
print(brw[["var5_brw_wohnen_niveau","var5_brw_wohnen_niveau_vs_bez","var5_brw_wohnen_niveau_vs_berlin",
           "var5_brw_wohnen_trend","var5_brw_wohnen_trend_vs_bez","var5_brw_wohnen_trend_vs_berlin"]].describe())

Valid niveau PLR: 540
Reliable PLR (cov>=0.5): 398
Trend reference PLR (reliable, no outlier): 540
Trend outliers: 1

Std-Check (sollte ~1.0 sein):
var5_brw_wohnen_niveau_vs_bez       0.989743
var5_brw_wohnen_niveau_vs_berlin    1.000000
var5_brw_wohnen_trend_vs_bez        0.989743
var5_brw_wohnen_trend_vs_berlin     1.000000
dtype: float64

WOHNEN:
       var5_brw_wohnen_niveau  var5_brw_wohnen_niveau_vs_bez  \
count              540.000000                   5.400000e+02   
mean              1608.003833                   3.289550e-17   
std               1489.554298                   9.897433e-01   
min                256.493697                  -1.927554e+00   
25%                569.960980                  -7.353124e-01   
50%                875.355040                  -2.812589e-01   
75%               2387.390313                   5.039619e-01   
max              11999.860612                   5.487267e+00   

       var5_brw_wohnen_niveau_vs_berlin  var5_brw_wohnen_trend  \
count

In [16]:
#save the variables 
out_dir = Path("../../data/real_estate/final_variables")
cols_wohnen = [
    "plr_id", "plr_name", "bez",
    "var5_brw_wohnen_niveau",
    "var5_brw_wohnen_niveau_vs_bez",
    "var5_brw_wohnen_niveau_vs_berlin",
    "var5_brw_wohnen_trend",
    "var5_brw_wohnen_trend_vs_bez",
    "var5_brw_wohnen_trend_vs_berlin",
    "var5_brw_wohnen_trend_outlier",
    "cov_wohnen_2025",
]
brw[cols_wohnen].to_csv(out_dir / "var5_brw_wohnen_features.csv", index=False)
print("gespeichert:", brw[cols_wohnen].shape)

gespeichert: (542, 11)


### Results Data Load Variable Land Value 

Residential land values were aggregated from the Berlin BRW WFS layers (2021–2025) onto the 542 PLR as area-weighted means, retaining per-land-use coverage so reliability could be judged on the residential footprint specifically. Only residential (wohnen) land is carried forward: commercial was dropped because too few PLR (28 at a 50% coverage threshold) had a substantively commercial character to support stable per-district standardization, and mixed/core and other categories are not conceptually central to gentrification.

The level (niveau) is the 2025 residential BRW, available for 540 PLR, averaging about €1,608/m² (median €875, range €256–12,000) — strongly right-skewed, as land prices typically are. The trend is the log-ratio ln(2025/2021), which replaces an earlier percent-change measure that produced extreme artefacts on PLR with very low 2021 baselines (e.g. Gartenfeld, a redevelopment area). The log transform compresses these: the trend now ranges roughly −0.69 to +1.90 (mean −0.22), and one PLR remains flagged as a trend outlier.

Both measures are standardized district-relative (vs_bez) and city-relative (vs_berlin); all four z-score families are well-formed (means ≈ 0, std ≈ 1.0). A reliability flag based on residential coverage (≥ 50% of PLR area; 398 PLR qualify) and the trend-outlier flag are retained in the output so the clustering stage can down-weight or exclude thin-basis and extreme PLR rather than having those decisions baked into the variable.

## Variable 6 AirBNB Dichte pro PLR

These cells load the Airbnb listings, reduces them to relevant columns, and converts the coordinates into geo-points that are assigned to the planning areas via a spatial join. Listings per PLR are counted (total and entire-home only), normalised by the housing stock into density variables (per 1,000 dwellings), and saved as a GeoPackage. The comments flag small-denominator artefacts and the strong right skew, which should be addressed in the modelling phase (e.g. log transformation).

The data can be found here: https://insideairbnb.com/get-the-data/

The file listings.csv.gz is downloaded and renamed to be distinguishable in the raw variable folder. 

In [17]:
#load data 
AirBNB_data = pd.read_csv("../../data/real_estate/raw/AirBNB_listings_raw.csv")

print(AirBNB_data.shape)
print(AirBNB_data.columns)

#Ansicht Aufteilung nach room_type - Entscheidung: eine Variable nur entire-Home / eine Variable alle listings pro PLR 
AirBNB_data["room_type"].value_counts()

#cut the unessential variables out 
data_AirBNB = AirBNB_data[['id', 'host_listings_count', 'latitude', 'longitude', 'neighbourhood_cleansed', 'room_type', 'property_type', 'minimum_nights', 'availability_365', 'number_of_reviews_ltm', 'calculated_host_listings_count',
'price', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'license']]

data_AirBNB.shape

#make a new variable from long/lat to a point on the Koordinatensystem (geopandas)
data_AirBNB_geo = gpd.GeoDataFrame(
    data_AirBNB,
    geometry=gpd.points_from_xy(data_AirBNB["longitude"], data_AirBNB["latitude"]),
    crs="EPSG:4326"
)

data_AirBNB_geo.head()

#Laden der Planungsräume 
plr = gpd.read_file("../../data/real_estate/Geodata/PLR_raw.geojson")
print(plr.shape)
print(plr.columns)
print(plr.crs)

#Anpassung AirBNB Koordinatensystem auf Berlin Koordinatensystem
data_AirBNB_geo = data_AirBNB_geo.to_crs(25833)
print(data_AirBNB_geo.crs)

#join AIRBNB points to PLR --> welches Airbnb (alle listings) liegt in welchem PLR? 
airbnb_plr = gpd.sjoin(data_AirBNB_geo, plr, predicate="within")

#check 
print("Listings vor join:", len(data_AirBNB_geo), "nach join:", len(airbnb_plr))

#airbnb_plr.shape
airbnb_plr.columns
airbnb_plr[["id_left", "plr_id", "plr_name"]].head()

#alle Listings pro PLR zählen
counts_all = airbnb_plr.groupby("plr_id").size().rename("count_all")

#var2: nur die Entire homes 
counts_entire = (airbnb_plr[airbnb_plr["room_type"] == "Entire home/apt"]
                 .groupby("plr_id").size().rename("count_entire"))

#two columns in one dataset 
plr_counts = (plr
              .merge(counts_all, on="plr_id", how="left")
              .merge(counts_entire, on="plr_id", how="left"))

#35 Missing values - replace with 0 when no airbnb points in this PLR 
plr_counts[["count_all", "count_entire"]] = (
    plr_counts[["count_all", "count_entire"]].fillna(0).astype(int)
)

# Nur die Spalten behalten, die wir wirklich brauchen
plr_counts = plr_counts[[
    "plr_id",        
    "plr_name",      
    "bez",           
    "geometry",      
    "count_all",
    "count_entire",
]]

#Laden Wohnungsbestand / PLR um Indikator zu bauen ; str damit 0 am Anfang des PLRs bleibt 
wohnungen = pd.read_excel("../../data/real_estate/raw/Anzahl_whg_PLR.xlsx", skiprows=2, dtype={0: str, 1: str})
wohnungen.columns = ["plr_id", "wohnungen"]
wohnungen = wohnungen[wohnungen["plr_id"].str.match(r"^\d{8}$", na=False)]
wohnungen["wohnungen"] = pd.to_numeric(
    wohnungen["wohnungen"].astype(str).str.replace(".", "", regex=False),
    errors="coerce")

wohnungen.head()

#check index before merging the two datasets together 
print(wohnungen["plr_id"].iloc[0], type(wohnungen["plr_id"].iloc[0]))
print(plr_counts["plr_id"].iloc[0], type(plr_counts["plr_id"].iloc[0]))

#merge the data 
plr_counts = plr_counts.merge(wohnungen, on="plr_id", how="left")

#safety guard 
assert plr_counts["wohnungen"].max() > 1000, "wohnungen zu klein — Tausenderpunkt-Bug?"

print(plr_counts["wohnungen"].isna().sum())   
print(plr_counts.head())

#Anzahl Wohnungen ist als text gespeichert, für division braucht es zahl 
print(plr_counts["wohnungen"].dtype)          
print(plr_counts["wohnungen"].isna().sum())  

#two PLRs dont have a number - both make sense to stay as NaNs as they dont have apartments 
print(plr_counts[plr_counts["wohnungen"].isna()][["plr_name", "count_all", "count_entire"]])

#create Variables for Dichte-entirehomes / dichte-all 

plr_counts["dichte_all"] = np.where(
    plr_counts["wohnungen"] > 0,
    plr_counts["count_all"] / plr_counts["wohnungen"] * 1000,
    np.nan
)

plr_counts["dichte_entire"] = np.where(
    plr_counts["wohnungen"] > 0,
    plr_counts["count_entire"] / plr_counts["wohnungen"] * 1000,
    np.nan
)


#have a look at the data now 
print(plr_counts[["count_all", "count_entire", "wohnungen",
                  "dichte_all", "dichte_entire"]].describe())

print(plr_counts.sort_values("dichte_entire", ascending=False)
      [["plr_name", "count_entire", "wohnungen", "dichte_entire"]].head(10))

#save the variables as one geofile 
plr_counts.to_file("../../data/real_estate/Geodata/var6_DichteAirBNB.gpkg", driver="GPKG")

#to reload later 
var6 = gpd.read_file("../../data/real_estate/Geodata/var6_DichteAirBNB.gpkg")

(14274, 79)
Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_a

### Results Data Load AirBNB Density

The raw file holds 14,274 Airbnb listings across 79 columns. Both datasets align on Berlin's usual coordination system EPSG:25833 with matching string PLR IDs, and the spatial join retained all 14,274 listings (none fell outside Berlin's PLR coverage). Listings were counted per PLR — both total and entire-home only — and divided by the residential stock to give density per 1,000 apartments. The measure covers all 542 PLR; two (Pankower Tor, Landweg) have no apartment base and are kept as NaN at this stage. 

Entire-home density is strongly right-skewed (median ≈ 2.0, mean ≈ 4.5, max ≈ 60.7 per 1,000 apartments), with the highest values in central tourist PLR. Small-denominator PLR such as Wriezener Bahnhof (60.7, on only 527 apartments) sit at the top on a thin residential base and react sensitively to a few listings — motivating a log transform and a minimum-denominator caveat at the clustering stage.


## Variable 7 Buildings' Age as Proxy for Modernisation Potential 

These cells read the dwellings' building-age classes from the census excel file, renames the eight class columns, and filters to valid PLR IDs. Missing values (“–”) become true zeros, and the two most recent classes are merged into one bin (y2011_plus).

Data can be found here: https://www.statistik-berlin-brandenburg.de/zensus22/lokale-daten-berlin

In [18]:
#helper for German-formatted numbers (thousands dots, dash = zero)
def zu_zahl(s):
    s = s.astype(str).str.strip().str.replace(".", "", regex=False)
    s = s.replace({"–": "0", "-": "0", "": np.nan, "nan": np.nan})
    return pd.to_numeric(s, errors="coerce")

#load (always fresh from file → idempotent)
baualter = pd.read_excel(
    "../../data/real_estate/raw/Zensus2022vollständigPLR.xlsx",
    sheet_name="4.3 Wohnungen",
    skiprows=7,
    dtype={0: str},
)

#columns: 0=PLR, 1=Insgesamt, 2..9 = the eight Baualter classes
baualter = baualter.iloc[:, list(range(10))]
baualter.columns = [
    "plr_id", "wohnungen_gesamt",
    "vor_1919", "y1919_1948", "y1949_1978", "y1979_1990",
    "y1991_2000", "y2001_2010", "y2011_2019", "y2020_plus",
]

#keep only real PLR rows (drops the "Insgesamt" summary row)
baualter = baualter[baualter["plr_id"].str.match(r"^\d{8}$", na=False)].reset_index(drop=True)

#clean classes: dash → 0, strip thousands dots; a genuine 0 is correct here
klassen = ["vor_1919", "y1919_1948", "y1949_1978", "y1979_1990",
           "y1991_2000", "y2001_2010", "y2011_2019", "y2020_plus"]
for c in klassen:
    baualter[c] = zu_zahl(baualter[c]).fillna(0)

#clean total: strip dots, keep NaN for genuinely missing (do NOT fillna)
baualter["wohnungen_gesamt"] = pd.to_numeric(
    baualter["wohnungen_gesamt"].astype(str).str.replace(".", "", regex=False),
    errors="coerce")

#merge the two youngest classes into one bin (idempotent guard)
if "y2011_2019" in baualter.columns:
    baualter["y2011_plus"] = baualter["y2011_2019"] + baualter["y2020_plus"]
    baualter = baualter.drop(columns=["y2011_2019", "y2020_plus"])

#checks
assert baualter["wohnungen_gesamt"].max() > 1000, "Tausenderpunkt-Bug? Total zu klein"

klassen_neu = ["vor_1919", "y1919_1948", "y1949_1978", "y1979_1990",
               "y1991_2000", "y2001_2010", "y2011_plus"]
ratio = baualter[klassen_neu].sum(axis=1) / baualter["wohnungen_gesamt"]

print("shape:", baualter.shape)
print("wohnungen_gesamt max:", baualter["wohnungen_gesamt"].max())
print("\nKlassen-Summe / Gesamt (sollte ≈ 1.0):")
print(ratio.describe())
print("\n", baualter.head())

shape: (541, 9)
wohnungen_gesamt max: 10175.0

Klassen-Summe / Gesamt (sollte ≈ 1.0):
count    540.000000
mean       1.001278
std        0.024235
min        0.980392
25%        0.999174
50%        1.000000
75%        1.000670
max        1.500000
dtype: float64

      plr_id  wohnungen_gesamt  vor_1919  y1919_1948  y1949_1978  y1979_1990  \
0  01100101            2065.0       244          17        1160         307   
1  01100102            1163.0       266          18          24          97   
2  01100103            2897.0       938         147         741         545   
3  01100104            2498.0       694          77         810         432   
4  01100205            1362.0       230           6          57         417   

   y1991_2000  y2001_2010  y2011_plus  
0         130           0         212  
1         412         264          82  
2         144         108         269  
3          11         188         286  
4         307         238         108  


/Users/lauragies/Documents/Job/NF_Bootcamp/capstone-kiezkeeper/.venv/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [19]:
baualter["ratio"] = baualter[klassen_neu].sum(axis=1) / baualter["wohnungen_gesamt"]
print(baualter.nlargest(3, "ratio")[["plr_id","wohnungen_gesamt"]+klassen_neu])
print(baualter.nsmallest(3, "ratio")[["plr_id","wohnungen_gesamt"]])
baualter = baualter.drop(columns="ratio")

       plr_id  wohnungen_gesamt  vor_1919  y1919_1948  y1949_1978  y1979_1990  \
427  10100205              10.0         3           3           0           3   
234  05300838               8.0         0           6           4           0   
168  04400725              72.0        50           3          19           0   

     y1991_2000  y2001_2010  y2011_plus  
427           0           0           6  
234           0           0           0  
168           3           0           0  
       plr_id  wohnungen_gesamt
486  11300723             459.0
101  03300517             516.0
361  08200728             438.0


In [20]:
# Anteile bilden (fürs Clustering, da PLR-Größe sonst dominiert)
final_bins = ["vor_1919", "y1919_1948", "y1949_1978",
              "y1979_1990", "y1991_2000", "y2001_2010", "y2011_plus"]
for c in final_bins:
    baualter[f"anteil_{c}"] = baualter[c] / baualter["wohnungen_gesamt"]

# reliability flag: shares from very few dwellings (or unknown total) are unstable
baualter["baualter_unsicher"] = (
    (baualter["wohnungen_gesamt"] < 50) | baualter["wohnungen_gesamt"].isna()
).astype("Int64")
print("PLR geflaggt (unsicher):", int(baualter["baualter_unsicher"].sum()))

#speichern
out_dir = Path("../../data/real_estate/final_variables")
out_dir.mkdir(parents=True, exist_ok=True)
baualter.to_csv(out_dir / "var7_baualter.csv", index=False)
var7 = pd.read_csv(out_dir / "var7_baualter.csv", dtype={"plr_id": str})

PLR geflaggt (unsicher): 4


In [21]:
#check coverage of Wohnungen / Baualterbin to find a threshold 
for schwelle in [30, 50, 100]:
    n = (baualter["wohnungen_gesamt"] < schwelle).sum()
    n_na = baualter["wohnungen_gesamt"].isna().sum()
    print(f"< {schwelle} Wohnungen: {n} PLR geflaggt (+ {n_na} NaN-Gesamt)")

< 30 Wohnungen: 3 PLR geflaggt (+ 1 NaN-Gesamt)
< 50 Wohnungen: 3 PLR geflaggt (+ 1 NaN-Gesamt)
< 100 Wohnungen: 4 PLR geflaggt (+ 1 NaN-Gesamt)


In [22]:
#check with PLR is missing - the same than in the Zensus 
print(set(var6["plr_id"]) - set(var7["plr_id"]))

{'03400831'}


### Results Data Load Buildings' Age 

Building-age structure was taken from the Zensus 2022 dwelling table (table 4.3 Wohnungen) at planning area level. The eight original age classes were cleaned (German thousands separators stripped, the Zensus dash read as a true zero) and the two youngest classes (2011–2019 and 2020+) merged into a single y2011_plus bin, since the most recent construction is small and noisy this soon after 2020. The cleaned class counts sum to the stated dwelling total per PLR (ratio ≈ 1.0; minor deviations only on very small PLR, reflecting Zensus cell rounding), confirming the parse is correct.

Each class was converted to a share of total dwellings, giving a seven-bin age profile per PLR that is comparable across PLR of different sizes. The full profile is retained deliberately: Sanierungspotenzial is understood to span both period-building refurbishment (pre-1948 stock) and energetic retrofit (mid-century stock), so all age bins carry relevant signal rather than collapsing to a single pre-war proxy. However, in the EDA stage this decision needs to be revisited particularly concerning the clustering of the variables in Module 1. 

The measure covers 541 of 542 PLR; one PLR (03400831) is absent from the Zensus source and carries as NaN, consistent with the other Zensus-based variables. A reliability flag ("baualter_unsicher") marks 4 PLR whose shares rest on an unstable base — three with fewer than 50 dwellings and one with an unknown total — so the clustering stage can down-weight or exclude them. The shares are saved on their native 0–1 scale; standardisation is deferred to the clustering stage, where it will be applied uniformly across all variables.